In [1]:
import pandas as pd
import numpy as np 
from scipy import stats as stat
from scipy.stats import bootstrap as bootstrap
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn import catplot
import random as rd
import os
from sklearn.preprocessing import StandardScaler as ss
from sklearn.decomposition import PCA as pca
import math
from scipy.stats import linregress
import mne
%matplotlib qt
import os
from itertools import compress

import mne
from mne import datasets
import numpy as np
from nilearn.plotting import plot_design_matrix

import mne_nirs
import mne_bids
# Import StatsModels
import statsmodels.formula.api as smf
import matplotlib as mpl

# Import MNE processing
from mne.preprocessing.nirs import beer_lambert_law, optical_density

# Import MNE-BIDS processing
from mne_bids import BIDSPath, get_entity_vals, read_raw_bids,stats

from mne_nirs.channels import get_long_channels, get_short_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.io.fold import fold_channel_specificity

# Import MNE-NIRS processing
from mne_nirs.statistics import run_glm, statsmodels_to_results
from mne_nirs.visualisation import plot_glm_group_topo, plot_glm_surface_projection 
import os.path as op

from mne.datasets.testing import data_path
from mne.viz import set_3d_view

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm
# Import common libraries
from collections import defaultdict
from copy import deepcopy
from itertools import compress
from pprint import pprint

# Import Plotting Library
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import StatsModels
import statsmodels.formula.api as smf
from mne import Epochs, events_from_annotations, set_log_level
from mne.preprocessing.nirs import (
    beer_lambert_law,
    optical_density,
    scalp_coupling_index,
    temporal_derivative_distribution_repair,
)

# Import MNE processing
from mne.viz import plot_compare_evokeds

# Import MNE-BIDS processing
from mne_bids import BIDSPath, read_raw_bids

# Import MNE-NIRS processing
from mne_nirs.channels import get_long_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.signal_enhancement import enhance_negative_correlation

C:\Users\pkell\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [3]:
## Load in Functional Connectivity Data From MATLAB 
file = r"C:\Users\pkell\OneDrive\Documents\MATLAB\EI_ISC_group_functional_conn_ROI_tvr.csv"
df = pd.read_csv(file)

In [ ]:
df['condition']

In [ ]:
# Filtering the data by condition (if you have different conditions as opposed to task vs rest connectivity)
df1 = df[df["condition"] == 'positive']
df2 = df[df["condition"] == 'negative']

## Correlate vector of t values from 1st order to T vector from second order

from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# # --- Ensure data is sorted consistently ---
# df1 = df_cond1.sort_values(by=['ROI_X', 'ROI_Y', 'chOrigin', 'chDest']).reset_index(drop=True)
# df2 = df_cond2.sort_values(by=['ROI_X', 'ROI_Y', 'chOrigin', 'chDest']).reset_index(drop=True)

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df1 = df1[~df1['DetectorOrigin'].isin(bad_detectors)]
df2 = df2[~df2['DetectorOrigin'].isin(bad_detectors)]
df1 = df1[~df1['DetectorDest'].isin(bad_detectors)]
df2 = df2[~df2['DetectorDest'].isin(bad_detectors)]

# # --- Drop Specific Source-Detector Pairs ---
# df1 = df1[~((df1['source'] == 14) & (df1['detector'] == 13))]
# df2 = df2[~((df2['source'] == 16) & (df2['detector'] == 12))]

# --- Remove mirror-flipped channel pairs ---
def make_pair_id(df):
    return df.apply(
        lambda row: tuple(sorted([
            (row['SourceOrigin'], row['DetectorOrigin']),
            (row['SourceDest'], row['DetectorDest'])
        ])), axis=1)

df1['pair_id'] = make_pair_id(df1)
df2['pair_id'] = make_pair_id(df2)

# Keep only one direction per pair
df1 = df1.drop_duplicates(subset='pair_id')
df2 = df2.drop_duplicates(subset='pair_id')

# Drop the helper column
df1 = df1.drop(columns='pair_id')
df2 = df2.drop(columns='pair_id')

# --- Create the r-value vectors ---
r_vec_cond1 = df1['R'].values
r_vec_cond2 = df2['R'].values

# --- Double-check that vectors have the same length ---
if len(r_vec_cond1) != len(r_vec_cond2):
    raise ValueError("Condition 1 and Condition 2 vectors do not have the same length! Check preprocessing.")
else:
    print(f"✅ Matched number of channel pairs: {len(r_vec_cond1)}")

# --- Remove pairs where either condition has NaN ---
valid_mask = np.isfinite(r_vec_cond1) & np.isfinite(r_vec_cond2)
r_vec_cond1_clean = r_vec_cond1[valid_mask]
r_vec_cond2_clean = r_vec_cond2[valid_mask]

print(f"✅ Number of valid (non-NaN) pairs: {len(r_vec_cond1_clean)}")

# --- Correlate the cleaned vectors ---
r_across_conditions, p_value_across_conditions = pearsonr(r_vec_cond1_clean, r_vec_cond2_clean)

# --- Print the results ---
print(f"📈 Pearson correlation between Condition 1 and Condition 2 channel-level r values (after removing NaNs):")
print(f"r = {r_across_conditions:.3f}, p = {p_value_across_conditions:.30f}")

# --- Optional: Visualize ---
plt.figure(figsize=(7, 6))
sns.regplot(x=r_vec_cond1_clean, y=r_vec_cond2_clean, scatter_kws={'s':30, 'alpha':0.7})

plt.title('Channel-level FC Correlation: Condition 1 vs Condition 2', fontsize=14)
plt.xlabel('Condition 1 r-values')
plt.ylabel('Condition 2 r-values')
plt.annotate(f"r = {r_across_conditions:.2f}\np = {p_value_across_conditions:.4f}",
             xy=(0.05, 0.95), xycoords='axes fraction',
             ha='left', va='top', fontsize=11,
             bbox=dict(boxstyle="round", facecolor='white', edgecolor='gray', alpha=0.8))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [5]:
## Run this Formatting Code 

df_hbo = df[df['TypeOrigin'] == 'hbo']
df_hbo = df_hbo[df_hbo['TypeDest'] == 'hbo']

# --- Drop Rows with Detector in {16–23} ---
bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df_hbo = df_hbo[~df_hbo['DetectorOrigin'].isin(bad_detectors)]
df_hbo = df_hbo[~df_hbo['DetectorDest'].isin(bad_detectors)]

# Create a new column 'chOrigin' with the structure "S<SourceOrigin>_D<DetectorOrigin> <TypeOrigin>"
df_hbo['chOrigin'] = (
    'S' + df_hbo['SourceOrigin'].astype(str) +
    '_D' + df_hbo['DetectorOrigin'].astype(str) +
    ' ' + df_hbo['TypeOrigin']
)

# Create a new column 'chDest' with the structure "S<SourceDest>_D<DetectorDest> <TypeDest>"
df_hbo['chDest'] = (
    'S' + df_hbo['SourceDest'].astype(str) +
    '_D' + df_hbo['DetectorDest'].astype(str) +
    ' ' + df_hbo['TypeDest']
)

# Optionally, if you no longer need the original columns, you can drop them:
df_hbo = df_hbo.drop(columns=[
    'SourceOrigin', 'DetectorOrigin', 'TypeOrigin',
    'SourceDest', 'DetectorDest', 'TypeDest'
])

# Inspect the modified dataframe
df_hbo


,condition,R,Z,t,pvalue,qvalue,chOrigin,chDest
0,Task,NaN,NaN,NaN,1.000000e+00,1.000000e+00,S2_D2 hbo,S2_D2 hbo
1,Task,0.478484,0.521017,13.137782,1.942813e-14,1.424730e-13,S2_D2 hbo,S4_D3 hbo
2,Task,0.432751,0.463277,13.388818,1.160634e-14,1.094312e-13,S2_D2 hbo,S3_D1 hbo
3,Task,0.562124,0.635932,12.585676,6.182147e-14,2.914440e-13,S2_D2 hbo,S1_D15 hbo
4,Task,0.264284,0.270709,9.477982,8.277578e-11,1.162383e-10,S2_D2 hbo,S11_D10 hbo
...,...,...,...,...,...,...,...,...
139,Task,0.205424,0.208389,10.687660,4.346035e-12,8.692069e-12,S15_D14 hbo,S8_D7 hbo
140,Task,0.155844,0.157125,8.232651,2.099157e-09,2.388696e-09,S15_D14 hbo,S9_D6 hbo
141,Task,0.219320,0.222942,11.660694,4.650783e-13,1.334572e-12,S15_D14 hbo,S6_D15 hbo
142,Task,0.412629,0.438776,7.479114,1.632343e-08,1.657456e-08,S15_D14 hbo,S14_D11 hbo


In [7]:
# # Define ROIs (HbO only)
# roi_definitions = {
#     "Left RLPFC": ['S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'],
#     "Left VLPFC": ['S3_D1 hbo', 'S3_D3 hbo'],
#     "Left DLPFC": ['S2_D3 hbo', 'S4_D3 hbo', 'S4_D2 hbo'],
#     "Left OFC": ['S1_D15 hbo'],
#     "Left STS": ['S5_D4 hbo','S5_D10 hbo','S11_D4 hbo', 'S11_D10 hbo', 'S12_D9 hbo', 'S12_D10 hbo'],
#     "Left TPJ": ['S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo','S13_D9 hbo'],
#     "Right RLPFC": ['S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo'],
#     "Right DLPFC": ['S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo'],
#     "Right VLPFC": ['S9_D7 hbo', 'S9_D6 hbo'],
#     "Right OFC": ['S6_D15 hbo'],
#     "Right STS": ['S10_D13 hbo','S10_D11 hbo','S14_D11 hbo', 'S14_D14 hbo','S15_D11 hbo','S15_D13 hbo'],
#     "Right TPJ": ['S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo','S16_D14 hbo']
# }

# Define ROIs (HbO only)
roi_definitions = {
    "L-RLPFC": ['S2_D2 hbo'],
    "L-VLPFC": ['S3_D1 hbo'],
    "L-DLPFC": ['S4_D3 hbo'],
    "L-OFC": ['S1_D15 hbo'],
    "L-STS": ['S11_D10 hbo'],
    "L-TPJ": ['S11_D9 hbo'],
    "R-RLPFC": ['S7_D5 hbo'],
    "R-DLPFC": ['S8_D7 hbo'],
    "R-VLPFC": ['S9_D6 hbo'],
    "R-OFC": ['S6_D15 hbo'],
    "R-STS": ['S14_D11 hbo'],
    "R-TPJ": ['S15_D14 hbo']
}


# Optional: Function to normalize channel names (lower-case and stripped)
def normalize_channel(ch):
    return ch.strip().lower()

# Function to look up ROI for a given channel name.
def get_roi(channel_name, roi_definitions):
    # Normalize the channel name for matching
    ch_norm = normalize_channel(channel_name)
    for roi, channels in roi_definitions.items():
        # Normalize each channel name for robust matching
        norm_channels = [normalize_channel(ch) for ch in channels]
        if ch_norm in norm_channels:
            return roi
    return "Unassigned"

# Assume df_hbo is your dataframe that now contains the combined columns "chOrigin" and "chDest".
# (For example, you previously computed these as:
#  df_hbo['chOrigin'] = 'S' + df_hbo['SourceOrigin'].astype(str) + '_D' + df_hbo['DetectorOrigin'].astype(str) + ' ' + df_hbo['TypeOrigin']
#  df_hbo['chDest'] = 'S' + df_hbo['SourceDest'].astype(str) + '_D' + df_hbo['DetectorDest'].astype(str) + ' ' + df_hbo['TypeDest']
#)

# Create new ROI columns by applying the lookup function on the combined channel columns.
df_hbo = df_hbo.copy()  # Create a copy if needed

# Use the new column names for ROI lookup
df_hbo['ROI_X'] = df_hbo['chOrigin'].apply(lambda ch: get_roi(ch, roi_definitions))
df_hbo['ROI_Y'] = df_hbo['chDest'].apply(lambda ch: get_roi(ch, roi_definitions))

# Convert these new ROI columns to categorical variables.
df_hbo['ROI_X'] = df_hbo['ROI_X'].astype('category')
df_hbo['ROI_Y'] = df_hbo['ROI_Y'].astype('category')

## this section is unneccessary keep it commented out 

# # Compute the global r for each subject and add it as a new column
# df_hbo['global_r'] = df_hbo.groupby('subject')['R'].transform('mean')

# # For each row (channel pair), compute the deviation: the difference between the row's R value and the subject's global r
# df_hbo['deviation'] = df_hbo['R'] - df_hbo['global_r']

# #Compute the standard deviation of the deviations for each subject
# df_hbo['std_deviation'] = df_hbo.groupby('subject')['deviation'].transform('std')

# # Calculate the z-score for each channel pair (centered at the subject’s global r)
# df_hbo['z_score'] = df_hbo['deviation'] / df_hbo['std_deviation']
# Now, df_hbo contains your original data plus two new columns 'ROI_X' and 'ROI_Y'
# based on the channel names in 'chOrigin' and 'chDest'.
df_hbo

,condition,R,Z,t,pvalue,qvalue,chOrigin,chDest,ROI_X,ROI_Y
0,Task,NaN,NaN,NaN,1.000000e+00,1.000000e+00,S2_D2 hbo,S2_D2 hbo,L-RLPFC,L-RLPFC
1,Task,0.478484,0.521017,13.137782,1.942813e-14,1.424730e-13,S2_D2 hbo,S4_D3 hbo,L-RLPFC,L-DLPFC
2,Task,0.432751,0.463277,13.388818,1.160634e-14,1.094312e-13,S2_D2 hbo,S3_D1 hbo,L-RLPFC,L-VLPFC
3,Task,0.562124,0.635932,12.585676,6.182147e-14,2.914440e-13,S2_D2 hbo,S1_D15 hbo,L-RLPFC,L-OFC
4,Task,0.264284,0.270709,9.477982,8.277578e-11,1.162383e-10,S2_D2 hbo,S11_D10 hbo,L-RLPFC,L-STS
...,...,...,...,...,...,...,...,...,...,...
139,Task,0.205424,0.208389,10.687660,4.346035e-12,8.692069e-12,S15_D14 hbo,S8_D7 hbo,R-TPJ,R-DLPFC
140,Task,0.155844,0.157125,8.232651,2.099157e-09,2.388696e-09,S15_D14 hbo,S9_D6 hbo,R-TPJ,R-VLPFC
141,Task,0.219320,0.222942,11.660694,4.650783e-13,1.334572e-12,S15_D14 hbo,S6_D15 hbo,R-TPJ,R-OFC
142,Task,0.412629,0.438776,7.479114,1.632343e-08,1.657456e-08,S15_D14 hbo,S14_D11 hbo,R-TPJ,R-STS


In [ ]:
## Plot Channel x Channel Matrix with Color Coded Channels based on ROI 

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# Define ROIs (HbO only)
roi_definitions = {
    "Left RLPFC": ['S1_D1 hbo', 'S1_D2 hbo', 'S2_D1 hbo', 'S2_D2 hbo'],
    "Left VLPFC": ['S3_D1 hbo', 'S3_D3 hbo'],
    "Left DLPFC": ['S2_D3 hbo', 'S4_D3 hbo', 'S4_D2 hbo'],
    "Left OFC": ['S1_D15 hbo'],
    "Left STS": ['S5_D4 hbo','S5_D10 hbo','S11_D4 hbo', 'S11_D10 hbo', 'S12_D9 hbo', 'S12_D10 hbo'],
    "Left TPJ": ['S11_D9 hbo', 'S11_D8 hbo', 'S13_D8 hbo','S13_D9 hbo'],
    "Right RLPFC": ['S6_D5 hbo', 'S6_D6 hbo', 'S7_D5 hbo', 'S7_D6 hbo'],
    "Right DLPFC": ['S7_D7 hbo','S8_D5 hbo', 'S8_D7 hbo'],
    "Right VLPFC": ['S9_D7 hbo', 'S9_D6 hbo'],
    "Right OFC": ['S6_D15 hbo'],
    "Right STS": ['S10_D13 hbo','S10_D11 hbo','S14_D11 hbo', 'S14_D14 hbo','S15_D11 hbo','S15_D13 hbo'],
    "Right TPJ": ['S15_D12 hbo', 'S15_D14 hbo', 'S16_D12 hbo','S16_D14 hbo']
}

# --- Assign ROI colors ---
roi_colors = sns.color_palette("tab10", n_colors=len(roi_definitions))
roi_color_map = dict(zip(roi_definitions.keys(), roi_colors))

# --- Create channel → ROI map and ordered channel list ---
channel_to_roi = {}
ordered_channels = []
for roi, channels in roi_definitions.items():
    for ch in channels:
        channel_to_roi[ch] = roi
        ordered_channels.append(ch)

# --- Compute mean R values for all channel pairs ---
channel_fc_all = df_hbo.groupby(["chOrigin", "chDest"])["R"].mean().reset_index()

# --- Full grid of all channel pairs ---
full_index = pd.MultiIndex.from_product([ordered_channels, ordered_channels], names=["chOrigin", "chDest"])
full_df = pd.DataFrame(index=full_index).reset_index()
full_matrix = pd.merge(full_df, channel_fc_all, on=["chOrigin", "chDest"], how="left")

# --- Pivot matrix ---
heatmap_data = full_matrix.pivot(index="chOrigin", columns="chDest", values="R")
heatmap_data = heatmap_data.loc[ordered_channels, ordered_channels]

# --- Axis tick colors by ROI ---
x_colors = [roi_color_map[channel_to_roi[ch]] for ch in heatmap_data.columns]
y_colors = [roi_color_map[channel_to_roi[ch]] for ch in heatmap_data.index]

# --- Plot heatmap ---
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    heatmap_data,
    annot=False,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "Mean Functional Connectivity (R)"},
    ax=ax
)

# --- Color axis tick labels ---
for label, color in zip(ax.get_xticklabels(), x_colors):
    label.set_color(color)
for label, color in zip(ax.get_yticklabels(), y_colors):
    label.set_color(color)

# --- Titles and axis formatting ---
title = "Functional Connectivity Matrix"
plt.title(title, fontsize=14)
plt.xlabel("Destination Channel")
plt.ylabel("Origin Channel")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

# --- Create and place legend closer to y-axis ---
handles = [mpatches.Patch(color=color, label=roi) for roi, color in roi_color_map.items()]
fig.legend(
    handles,
    [roi for roi in roi_definitions.keys()],
    title="ROI",
    loc='center left',
    bbox_to_anchor=(0.05, 0.5),
    borderaxespad=0.5,
    frameon=True
)

# # --- Save figure ---
# save_path = r"C:\Users\pkell\OneDrive\Downloads"  # <-- EDIT THIS PATH
# plt.savefig(save_path, dpi=300)
# print(f"📁 Figure saved as: {save_path}")
    
plt.show()



In [15]:
## Plot Full ROI × ROI Correlation Matrix (Task vs Rest)

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# --- Convert ROI columns to uppercase to ensure matching ---
df_hbo['ROI_X'] = df_hbo['ROI_X'].str.upper()
df_hbo['ROI_Y'] = df_hbo['ROI_Y'].str.upper()

# --- Desired ROI Order (in uppercase) ---
roi_order = [
    'L-RLPFC', 'L-DLPFC', 'L-VLPFC',
    'R-RLPFC', 'R-DLPFC', 'R-VLPFC',
    'L-OFC', 'L-STS',
    'R-OFC', 'R-STS',
    'L-TPJ','R-TPJ',
]

# --- Compute mean R for each ROI pair ---
roi_fc_all = df_hbo.groupby(["ROI_X", "ROI_Y"])["R"].mean().reset_index()

# --- Create full grid of ROI combinations to ensure full matrix ---
roi_index = pd.MultiIndex.from_product([roi_order, roi_order], names=["ROI_X", "ROI_Y"])
full_roi_df = pd.DataFrame(index=roi_index).reset_index()

# --- Merge with actual data, keep missing pairs as NaN ---
full_matrix = pd.merge(full_roi_df, roi_fc_all, on=["ROI_X", "ROI_Y"], how="left")

# --- Pivot to square matrix ---
heatmap_data = full_matrix.pivot(index="ROI_X", columns="ROI_Y", values="R")

# --- Set order explicitly to preserve input ROI order ---
heatmap_data = heatmap_data.loc[roi_order, roi_order]

# --- Plot heatmap ---
plt.figure(figsize=(10, 8))
sns.heatmap(
    heatmap_data,
    annot=True,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "Mean Functional Connectivity (R)"}
)

plt.title("Functional Connectivity Matrix (ROI-Level)")
plt.xlabel("ROI_Y")
plt.ylabel("ROI_X")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# # --- Save figure ---
# save_path = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\36_group_FC_roi.png"  # <-- EDIT THIS PATH
# plt.savefig(save_path, dpi=300)
# print(f"📁 Figure saved as: {save_path}")
    
plt.show()

C:\Users\pkell\anaconda3\Lib\site-packages\seaborn\matrix.py:262: FutureWarning: Format strings passed to MaskedConstant are ignored, but in future may error or produce different behavior
  annotation = ("{:" + self.fmt + "}").format(val)


In [ ]:
full_matrix

In [25]:
# ===========================
# Load, preprocess, map ROIs, aggregate (Fisher z), and plot ROI × ROI FC
# with manual per-cell annotations (robust against seaborn annot bugs)
# ===========================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ---------- 1) LOAD ----------
file = r"C:\Users\pkell\OneDrive\Documents\MATLAB\EI_ISC_group_functional_conn_ROI_tvr.csv"
df = pd.read_csv(file)

# ---------- 2) PREPROCESS: keep HbO↔HbO & drop bad detectors ----------
df_hbo = df[(df['TypeOrigin'] == 'hbo') & (df['TypeDest'] == 'hbo')].copy()

bad_detectors = {16, 17, 18, 19, 20, 21, 22, 23}
df_hbo = df_hbo[~df_hbo['DetectorOrigin'].isin(bad_detectors)]
df_hbo = df_hbo[~df_hbo['DetectorDest'].isin(bad_detectors)]

# Create origin/dest channel labels like "S2_D2 hbo"
df_hbo['chOrigin'] = (
    'S' + df_hbo['SourceOrigin'].astype(str) +
    '_D' + df_hbo['DetectorOrigin'].astype(str) +
    ' ' + df_hbo['TypeOrigin']
)
df_hbo['chDest'] = (
    'S' + df_hbo['SourceDest'].astype(str) +
    '_D' + df_hbo['DetectorDest'].astype(str) +
    ' ' + df_hbo['TypeDest']
)

# Optional: drop now-redundant columns
df_hbo = df_hbo.drop(columns=[
    'SourceOrigin','DetectorOrigin','TypeOrigin',
    'SourceDest','DetectorDest','TypeDest'
], errors='ignore')

# ---------- 3) ROI MAPPING ----------
roi_definitions = {
    "L-RLPFC": ['S2_D2 hbo'],
    "L-VLPFC": ['S3_D1 hbo'],
    "L-DLPFC": ['S4_D3 hbo'],
    "L-OFC":   ['S1_D15 hbo'],
    "L-STS":   ['S11_D10 hbo'],
    "L-TPJ":   ['S11_D9 hbo'],
    "R-RLPFC": ['S7_D5 hbo'],
    "R-DLPFC": ['S8_D7 hbo'],
    "R-VLPFC": ['S9_D6 hbo'],
    "R-OFC":   ['S6_D15 hbo'],
    "R-STS":   ['S14_D11 hbo'],
    "R-TPJ":   ['S15_D14 hbo']
}

def normalize_channel(ch: str) -> str:
    return ch.strip().lower()

def get_roi(channel_name: str, roi_definitions: dict) -> str:
    ch_norm = normalize_channel(channel_name)
    for roi, channels in roi_definitions.items():
        norm_channels = [normalize_channel(c) for c in channels]
        if ch_norm in norm_channels:
            return roi
    return "Unassigned"

# Map to ROI_X/ROI_Y
df_hbo['ROI_X'] = df_hbo['chOrigin'].apply(lambda ch: get_roi(ch, roi_definitions))
df_hbo['ROI_Y'] = df_hbo['chDest'].apply(lambda ch: get_roi(ch, roi_definitions))

# Uppercase for consistent matching and cast to category
df_hbo['ROI_X'] = df_hbo['ROI_X'].str.upper().astype('category')
df_hbo['ROI_Y'] = df_hbo['ROI_Y'].str.upper().astype('category')

# Drop unassigned (prevents NaN-only pairs)
df_hbo = df_hbo[(df_hbo['ROI_X'] != 'UNASSIGNED') & (df_hbo['ROI_Y'] != 'UNASSIGNED')].copy()

## this section is unneccessary keep it commented out 
# # Compute the global r for each subject and add it as a new column
# df_hbo['global_r'] = df_hbo.groupby('subject')['R'].transform('mean')
# # For each row (channel pair), compute the deviation: the difference between the row's R value and the subject's global r
# df_hbo['deviation'] = df_hbo['R'] - df_hbo['global_r']
# #Compute the standard deviation of the deviations for each subject
# df_hbo['std_deviation'] = df_hbo.groupby('subject')['deviation'].transform('std')
# # Calculate the z-score for each channel pair (centered at the subject’s global r)
# df_hbo['z_score'] = df_hbo['deviation'] / df_hbo['std_deviation']

# ---------- 4) ORDER ----------
roi_order = [
    'L-RLPFC', 'L-DLPFC', 'L-VLPFC',
    'R-RLPFC', 'R-DLPFC', 'R-VLPFC',
    'L-OFC', 'L-STS',
    'R-OFC', 'R-STS',
    'L-TPJ', 'R-TPJ',
]
roi_order = [r.upper() for r in roi_order]

# ---------- 5) FISHER z AVERAGING PER ROI PAIR ----------
# Ensure numeric R and clip to avoid infs in arctanh
df_hbo['R'] = pd.to_numeric(df_hbo['R'], errors='coerce').clip(-0.999999, 0.999999)
df_hbo['Z'] = np.arctanh(df_hbo['R'])

# Average in z-space per (ROI_X, ROI_Y)
roi_fc_all = (
    df_hbo.groupby(['ROI_X','ROI_Y'], as_index=False)['Z']
          .mean()
)
# back-transform to r
roi_fc_all['R'] = np.tanh(roi_fc_all['Z'])
roi_fc_all = roi_fc_all.drop(columns='Z')

# ---------- 6) SYMMETRIZE (fill both triangles) ----------
swapped = roi_fc_all.rename(columns={'ROI_X':'ROI_Y','ROI_Y':'ROI_X'})
roi_fc_sym = pd.concat([roi_fc_all, swapped], ignore_index=True)
roi_fc_sym = (
    roi_fc_sym.groupby(['ROI_X','ROI_Y'], as_index=False)['R']
              .mean()
)

# ---------- 7) BUILD FULL GRID & PIVOT ----------
roi_index = pd.MultiIndex.from_product([roi_order, roi_order], names=["ROI_X", "ROI_Y"])
full_roi_df = pd.DataFrame(index=roi_index).reset_index()

full_matrix = pd.merge(full_roi_df, roi_fc_sym, on=['ROI_X','ROI_Y'], how='left')
heatmap_data = full_matrix.pivot(index='ROI_X', columns='ROI_Y', values='R')

# Reindex to your desired order
heatmap_data = heatmap_data.reindex(index=roi_order, columns=roi_order)

# Mirror any remaining gaps and ensure float dtype
heatmap_data = heatmap_data.combine_first(heatmap_data.T).astype(float)

# Optional: If you want the diagonal shown, uncomment:
# np.fill_diagonal(heatmap_data.values, 1.0)

# ---------- 8) PLOT (manual annotations) ----------
plt.figure(figsize=(10, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=False,                # <-- turn off seaborn's annot
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "Mean Functional Connectivity (R, Fisher z–avg)"},
    linewidths=0.5,
    linecolor="white"
)

# Manually overlay text on every finite cell (robust to seaborn annotation bugs)
vals = heatmap_data.values   # numpy array
nrows, ncols = vals.shape
for i in range(nrows):
    for j in range(ncols):
        v = vals[i, j]
        if np.isfinite(v):
            ax.text(
                j + 0.5, i + 0.5, f"{v:.2f}",
                ha="center", va="center",
                fontsize=8
            )

plt.title("Functional Connectivity Matrix (ROI-Level)")
plt.xlabel("ROI_Y")
plt.ylabel("ROI_X")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# # --- Save figure ---
# save_path = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v3\Figures\36_group_FC_roi.png"
# plt.savefig(save_path, dpi=300)
# print(f"📁 Figure saved as: {save_path}")

plt.show()


C:\Users\pkell\AppData\Local\Temp\ipykernel_27260\3140470338.py:104: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_hbo.groupby(['ROI_X','ROI_Y'], as_index=False)['Z']
C:\Users\pkell\AppData\Local\Temp\ipykernel_27260\3140470338.py:115: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  roi_fc_sym.groupby(['ROI_X','ROI_Y'], as_index=False)['R']
